In [ ]:
from __future__ import annotations

import os
import random
import subprocess
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import ImageFile
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
IMAGE_SIZE = 256
BATCH_SIZE = 16
NUM_WORKERS = 8
EPOCHS = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
EARLY_STOPPING_PATIENCE = 10

# input flag
INPUT_MODE = "rgb_baseline" 

HFRI_CUTOFF = 0.05     
WMBFR_CUTOFF = 0.10   

DATASET_HANDLE = "nocturnalnerd/balanced-dataset-701020"

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = WORK_DIR / "best_freqnet_rgb_baseline.pt"
EPOCH_CHECKPOINT_DIR = WORK_DIR / "epoch_checkpoints_rgb_baseline"
EPOCH_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

REPO_DIR = WORK_DIR / "FreqNet-DeepfakeDetection"
REPO_URL = "https://github.com/chuangchuangtan/FreqNet-DeepfakeDetection.git"


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


def resolve_dataset_root() -> Path:
    explicit_root = os.environ.get("DATASET_ROOT")
    if explicit_root:
        explicit_path = Path(explicit_root)
        if explicit_path.exists():
            return explicit_path
        raise FileNotFoundError(f"DATASET_ROOT={explicit_path} does not exist.")

    try:
        import kagglehub
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
        import kagglehub

    downloaded_root = Path(kagglehub.dataset_download(DATASET_HANDLE))
    direct = downloaded_root / "dataset_result_70_10_20_balanced"
    if direct.exists():
        return direct
    discovered = [p for p in downloaded_root.rglob("dataset_result_70_10_20_balanced") if p.is_dir()]
    if len(discovered) == 1:
        return discovered[0]
    if all((downloaded_root / split).is_dir() for split in ("train", "validation", "test")):
        return downloaded_root
    raise FileNotFoundError(f"Could not locate dataset_result_balanced under {downloaded_root}.")


seed_everything()
DATASET_ROOT = resolve_dataset_root()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Dataset:", DATASET_ROOT)
print("Input mode:", INPUT_MODE)
print("Checkpoint target:", CHECKPOINT_PATH)



# clone freqnet
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

freqnet_source = REPO_DIR / "networks" / "freqnet.py"
source_text = freqnet_source.read_text(encoding="utf-8")
patched_text = source_text.replace(".cuda()", "")
if patched_text != source_text:
    freqnet_source.write_text(patched_text, encoding="utf-8")

sys.path.insert(0, str(REPO_DIR))
from networks.freqnet import FreqNet  # noqa: E402

class FakePositiveImageFolder(ImageFolder):
    """ImageFolder that always returns real=0 and fake=1 plus the image path."""

    def __init__(self, root: Path, transform=None):
        super().__init__(str(root), transform=transform)
        if set(self.classes) != {"fake", "real"}:
            raise ValueError(f"Expected class folders {{'real', 'fake'}} at {root}; found {self.classes}.")

    def __getitem__(self, index: int):
        image, original_target = super().__getitem__(index)
        path, _ = self.samples[index]
        class_name = self.classes[original_target]
        label = torch.tensor(float(class_name == "fake"), dtype=torch.float32)
        return image, label, path


train_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
    ]
)
eval_transform = transforms.Compose(
    [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True), transforms.ToTensor()]
)

train_dataset = FakePositiveImageFolder(DATASET_ROOT / "train", train_transform)
val_dataset = FakePositiveImageFolder(DATASET_ROOT / "validation", eval_transform)
test_dataset = FakePositiveImageFolder(DATASET_ROOT / "test", eval_transform)


def class_counts(dataset: FakePositiveImageFolder) -> dict[str, int]:
    counts = Counter(dataset.classes[target] for _, target in dataset.samples)
    return {name: counts[name] for name in ("real", "fake")}


for split_name, split_dataset in (("train", train_dataset), ("validation", val_dataset), ("test", test_dataset)):
    print(f"{split_name:10s} {class_counts(split_dataset)}")

loader_generator = torch.Generator().manual_seed(SEED)
pin_memory = device.type == "cuda"

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=pin_memory, persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=4 if NUM_WORKERS > 0 else None, generator=loader_generator,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=pin_memory, persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=4 if NUM_WORKERS > 0 else None,
)

# depracated failed multi banf freqnet experiment
class MultiBandFrequencyFusion(nn.Module):
    def __init__(self, hfri_cutoff: float = HFRI_CUTOFF, wmbfr_cutoff: float = WMBFR_CUTOFF, eps: float = 1e-6):
        super().__init__()
        self.hfri_cutoff = hfri_cutoff
        self.wmbfr_cutoff = wmbfr_cutoff
        self.eps = eps

    def _standardize(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=(-2, -1), keepdim=True)
        std = x.std(dim=(-2, -1), keepdim=True).clamp_min(self.eps)
        return (x - mean) / std

    def _radius(self, h: int, w: int, x: torch.Tensor) -> torch.Tensor:
        yy = torch.linspace(-1.0, 1.0, h, device=x.device, dtype=x.dtype)
        xx = torch.linspace(-1.0, 1.0, w, device=x.device, dtype=x.dtype)
        grid_y, grid_x = torch.meshgrid(yy, xx, indexing="ij")
        radius = torch.sqrt(grid_x.square() + grid_y.square())
        radius = radius / radius.max().clamp_min(self.eps)
        return radius[None, None, :, :]

    def forward(self, rgb: torch.Tensor) -> torch.Tensor:
        if rgb.ndim != 4 or rgb.shape[1] != 3:
            raise ValueError(f"Expected [B,3,H,W] RGB input; got {tuple(rgb.shape)}")
        with torch.amp.autocast(device_type=rgb.device.type, enabled=False):
            rgb = rgb.float()
            gray = 0.2989 * rgb[:, 0:1] + 0.5870 * rgb[:, 1:2] + 0.1140 * rgb[:, 2:3]
            spectrum = torch.fft.fftshift(torch.fft.fft2(gray, dim=(-2, -1), norm="ortho"), dim=(-2, -1))
            radius = self._radius(gray.shape[-2], gray.shape[-1], gray)

            hfri_mask = (radius > self.hfri_cutoff).to(gray.dtype)
            hfri = torch.fft.ifft2(
                torch.fft.ifftshift(spectrum * hfri_mask, dim=(-2, -1)), dim=(-2, -1), norm="ortho"
            ).abs()

            k_f = -0.2 * radius.square() + 0.8 * radius - 0.05
            weighted = (torch.exp(k_f / 2.0) - 1.0) / radius.clamp_min(self.eps)
            wmbfr_mask = torch.where(radius > self.wmbfr_cutoff, weighted, torch.zeros_like(weighted))
            wmbfr = torch.fft.ifft2(
                torch.fft.ifftshift(spectrum * wmbfr_mask, dim=(-2, -1)), dim=(-2, -1), norm="ortho"
            ).abs()

            return torch.cat([self._standardize(gray), self._standardize(hfri), self._standardize(wmbfr)], dim=1)

# normal freqnet algo
class AmpSafeFreqNet(FreqNet):
    def __init__(self):
        super().__init__()
        with torch.no_grad():
            for weight, bias in (
                (self.weight1, self.bias1), (self.weight2, self.bias2),
                (self.weight3, self.bias3), (self.weight4, self.bias4),
            ):
                nn.init.kaiming_normal_(weight, mode="fan_in", nonlinearity="relu")
                nn.init.zeros_(bias)

    def hfreqWH(self, x: torch.Tensor, scale: int) -> torch.Tensor:
        assert scale > 2
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            x = torch.fft.fft2(x, norm="ortho")
            x = torch.fft.fftshift(x, dim=[-2, -1])
            _, _, h, w = x.shape
            mask = torch.ones((h, w), device=x.device, dtype=x.real.dtype)
            mask[h // 2 - h // scale : h // 2 + h // scale, w // 2 - w // scale : w // 2 + w // scale] = 0.0
            x = x * mask[None, None, :, :]
            x = torch.fft.ifftshift(x, dim=[-2, -1])
            x = torch.fft.ifft2(x, norm="ortho").real
            return F.relu(x, inplace=False)

    def hfreqC(self, x: torch.Tensor, scale: int) -> torch.Tensor:
        assert scale > 2
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            x = torch.fft.fft(x, dim=1, norm="ortho")
            x = torch.fft.fftshift(x, dim=1)
            _, c, _, _ = x.shape
            mask = torch.ones(c, device=x.device, dtype=x.real.dtype)
            mask[c // 2 - c // scale : c // 2 + c // scale] = 0.0
            x = x * mask[None, :, None, None]
            x = torch.fft.ifftshift(x, dim=1)
            x = torch.fft.ifft(x, dim=1, norm="ortho").real
            return F.relu(x, inplace=False)

    def _frequency_conv(self, x: torch.Tensor, real_conv: nn.Module, imag_conv: nn.Module) -> torch.Tensor:
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            x = torch.fft.fft2(x, norm="ortho")
            x = torch.fft.fftshift(x, dim=[-2, -1])
            x = torch.complex(real_conv(x.real), imag_conv(x.imag))
            x = torch.fft.ifftshift(x, dim=[-2, -1])
            x = torch.fft.ifft2(x, norm="ortho").real
            return F.relu(x, inplace=False)

    def _raw_projection(self, x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor, stride: int) -> torch.Tensor:
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            x = F.conv2d(x.float(), weight.float(), bias.float(), stride=stride, padding=0)
            return F.relu(x, inplace=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.hfreqWH(x, 4)
        x = self._raw_projection(x, self.weight1, self.bias1, stride=1)
        x = self.hfreqC(x, 4)
        x = self._frequency_conv(x, self.realconv1, self.imagconv1)

        x = self.hfreqWH(x, 4)
        x = self._raw_projection(x, self.weight2, self.bias2, stride=2)
        x = self.hfreqC(x, 4)
        x = self._frequency_conv(x, self.realconv2, self.imagconv2)

        x = self.maxpool(x)
        x = self.layer1(x)

        x = self.hfreqWH(x, 4)
        x = self._raw_projection(x, self.weight3, self.bias3, stride=1)
        x = self._frequency_conv(x, self.realconv3, self.imagconv3)

        x = self.hfreqWH(x, 4)
        x = self._raw_projection(x, self.weight4, self.bias4, stride=2)
        x = self._frequency_conv(x, self.realconv4, self.imagconv4)

        x = self.layer2(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        return self.fc1(x)


# run which experiment
class ExperimentModel(nn.Module):
    def __init__(self, input_mode: str = INPUT_MODE):
        super().__init__()
        if input_mode not in {"proposed", "rgb_baseline"}:
            raise ValueError("input_mode must be 'proposed' or 'rgb_baseline'")
        self.input_mode = input_mode
        self.fusion = MultiBandFrequencyFusion()
        self.backbone = AmpSafeFreqNet()
        self.register_buffer("imagenet_mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("imagenet_std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, rgb: torch.Tensor) -> torch.Tensor:
        if self.input_mode == "proposed":
            model_input = self.fusion(rgb)
        else:
            model_input = (rgb - self.imagenet_mean) / self.imagenet_std
        return self.backbone(model_input).flatten()


model = ExperimentModel(INPUT_MODE)
print("GPU count:", torch.cuda.device_count())
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

use_amp = device.type == "cuda"
amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
use_grad_scaler = use_amp and amp_dtype == torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=use_grad_scaler, init_scale=1024.0, growth_interval=2000)

print("AMP enabled:", use_amp, "| dtype:", amp_dtype if use_amp else "FP32")

def _forward_and_loss(images: torch.Tensor, labels: torch.Tensor, amp_enabled: bool):
    with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_enabled):
        logits = model(images)
    loss = criterion(logits.float(), labels.float())
    return logits, loss


def _finite_forward(images: torch.Tensor, labels: torch.Tensor):
    logits, loss = _forward_and_loss(images, labels, amp_enabled=use_amp)
    if torch.isfinite(logits).all() and torch.isfinite(loss):
        return logits, loss, False
    if use_amp:
        logits, loss = _forward_and_loss(images, labels, amp_enabled=False)
        if torch.isfinite(logits).all() and torch.isfinite(loss):
            return logits, loss, True
    bad_logits = int((~torch.isfinite(logits)).sum().item())
    raise FloatingPointError(
        f"Non-finite outputs remain even in a full-FP32 replay: bad_logits={bad_logits}, loss={loss.item()}."
    )


def run_training_epoch(loader: DataLoader) -> float:
    model.train()
    running_loss, seen = 0.0, 0
    fp32_fallback_batches, skipped_gradient_batches = 0, 0
    progress = tqdm(loader, desc="train", leave=False)

    for images, labels, _ in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        logits, loss, used_fp32_fallback = _finite_forward(images, labels)
        if used_fp32_fallback:
            fp32_fallback_batches += 1

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        if not torch.isfinite(grad_norm):
            if scaler.is_enabled():
                current_scale = scaler.get_scale()
                optimizer.zero_grad(set_to_none=True)
                scaler.update(new_scale=max(current_scale * 0.5, 1.0))
                skipped_gradient_batches += 1
                progress.set_postfix(loss=f"{running_loss / max(seen, 1):.4f}", skipped="grad-overflow")
                continue
            raise FloatingPointError(f"Non-finite gradient norm detected: {grad_norm.item()}")

        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.numel()
        running_loss += loss.item() * batch_size
        seen += batch_size
        progress.set_postfix(loss=f"{running_loss / seen:.4f}")

    if seen == 0:
        raise RuntimeError("No finite training batches were completed.")
    if fp32_fallback_batches or skipped_gradient_batches:
        print(f"AMP stability: {fp32_fallback_batches} FP32 replays; {skipped_gradient_batches} skipped steps.")
    return running_loss / seen


@torch.inference_mode()
def predict(loader: DataLoader):
    model.eval()
    running_loss, seen = 0.0, 0
    all_labels, all_probabilities, all_paths = [], [], []
    progress = tqdm(loader, desc="evaluate", leave=False)

    for images, labels, paths in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits, loss = _forward_and_loss(images, labels, amp_enabled=False)
        if not torch.isfinite(logits).all() or not torch.isfinite(loss):
            raise FloatingPointError("FP32 evaluation produced non-finite values.")
        probabilities = torch.sigmoid(logits.float())
        batch_size = labels.numel()
        running_loss += loss.item() * batch_size
        seen += batch_size
        all_labels.append(labels.detach().cpu().numpy().astype(np.int64))
        all_probabilities.append(probabilities.detach().cpu().numpy())
        all_paths.extend(paths)

    return running_loss / seen, np.concatenate(all_labels), np.concatenate(all_probabilities), all_paths

start_epoch = 1
best_val_auc = -np.inf
epochs_without_improvement = 0
history: list[dict[str, float]] = []

# resume mechanism
RESUME_CHECKPOINT_PATH = Path("/kaggle/input/datasets/mikeeee54/latest-rgb-baseline-in-progress1/latest_rgb_baseline_in_progress(1).pt")
LATEST_IN_PROGRESS_PATH = WORK_DIR / "latest_rgb_baseline_in_progress(1).pt"
if RESUME_CHECKPOINT_PATH.exists():
    print(f"Found an in-progress checkpoint at {RESUME_CHECKPOINT_PATH}, resuming.")
    checkpoint = torch.load(RESUME_CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    best_val_auc = checkpoint["best_val_roc_auc_so_far"]
    print(f"Resuming from epoch {start_epoch}, best val ROC-AUC so far: {best_val_auc:.5f}")

for epoch in range(start_epoch, EPOCHS + 1):
    print(f"\n{'=' * 70}\nEpoch {epoch}/{EPOCHS} [{INPUT_MODE}]\n{'=' * 70}")

    train_loss = run_training_epoch(train_loader)
    val_loss, val_y, val_probability, _ = predict(val_loader)
    val_auc = roc_auc_score(val_y, val_probability)
    scheduler.step(val_auc)
    current_lr = optimizer.param_groups[0]["lr"]

    history.append(
        {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_roc_auc": val_auc, "learning_rate": current_lr}
    )
    print(f"Train Loss : {train_loss:.6f}\nVal Loss   : {val_loss:.6f}\nVal ROC-AUC: {val_auc:.6f}\nLR         : {current_lr:.8f}")

    checkpoint_payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_roc_auc": val_auc,
        "learning_rate": current_lr,
        "input_mode": INPUT_MODE,
        "image_size": IMAGE_SIZE,
        "label_mapping": {"real": 0, "fake": 1},
        "best_val_roc_auc_so_far": max(best_val_auc, val_auc),
    }
    torch.save(checkpoint_payload, LATEST_IN_PROGRESS_PATH)
    torch.save(checkpoint_payload, EPOCH_CHECKPOINT_DIR / f"model_epoch_{epoch:03d}.pt")

    if val_auc > best_val_auc + 1e-4:
        best_val_auc = val_auc
        epochs_without_improvement = 0
        torch.save({**checkpoint_payload, "best_val_roc_auc": best_val_auc}, CHECKPOINT_PATH)
        print(f"Saved NEW BEST checkpoint: validation ROC-AUC={best_val_auc:.5f}")
    else:
        epochs_without_improvement += 1
        print(f"No ROC-AUC improvement ({epochs_without_improvement}/{EARLY_STOPPING_PATIENCE})")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch}.\nBest validation ROC-AUC: {best_val_auc:.5f}")
        break

history_df = pd.DataFrame(history)
print(history_df)
print(f"\nTraining completed. Best validation ROC-AUC: {best_val_auc:.5f}")
print(f"Best rgb_baseline checkpoint: {CHECKPOINT_PATH}")

Device: cuda
Dataset: /kaggle/input/datasets/nocturnalnerd/balanced-dataset-701020/dataset_result_70_10_20_balanced
Input mode: rgb_baseline
Checkpoint target: /kaggle/working/best_freqnet_rgb_baseline.pt


Cloning into '/kaggle/working/FreqNet-DeepfakeDetection'...


train      {'real': 19368, 'fake': 19368}
validation {'real': 2767, 'fake': 2767}
test       {'real': 5534, 'fake': 5534}
GPU count: 2
Using 2 GPUs


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


AMP enabled: True | dtype: torch.bfloat16
Found an in-progress checkpoint at /kaggle/input/datasets/mikeeee54/latest-rgb-baseline-in-progress1/latest_rgb_baseline_in_progress(1).pt, resuming.
Resuming from epoch 27, best val ROC-AUC so far: 0.98452

Epoch 27/30 [rgb_baseline]


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


... (8 outputs left out)

Train Loss : 0.106951
Val Loss   : 0.303083
Val ROC-AUC: 0.977698
LR         : 0.00010000
No ROC-AUC improvement (2/10)

Epoch 30/30 [rgb_baseline]


train:   0%|          | 0/2421 [00:00<?, ?it/s]

evaluate:   0%|          | 0/346 [00:00<?, ?it/s]

Train Loss : 0.106688
Val Loss   : 0.435898
Val ROC-AUC: 0.983203
LR         : 0.00005000
No ROC-AUC improvement (3/10)
   epoch  train_loss  val_loss  val_roc_auc  learning_rate
0     27    0.121438  0.269655     0.984639        0.00010
1     28    0.113791  0.264875     0.982638        0.00010
2     29    0.106951  0.303083     0.977698        0.00010
3     30    0.106688  0.435898     0.983203        0.00005

Training completed. Best validation ROC-AUC: 0.98464
Best rgb_baseline checkpoint: /kaggle/working/best_freqnet_rgb_baseline.pt
